In [ ]:
#llama lora로 학습
import torch
import os
from datasets import load_dataset, Dataset
from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    BitsAndBytesConfig
)
from peft import LoraConfig
# ★ SFTConfig 추가 임포트
from trl import SFTTrainer, SFTConfig 

import time, math
def bench_and_warn(trainer, dataset, batch_size, grad_accum, num_epochs, warmup_steps=10, max_hours=3):
    num_devices = max(1, torch.cuda.device_count())
    eff_batch = batch_size * grad_accum * num_devices
    n_samples = len(dataset)
    steps_per_epoch = math.ceil(n_samples / eff_batch) if eff_batch>0 else 0
    total_steps = steps_per_epoch * num_epochs
    print(f"Samples:{n_samples} eff_batch:{eff_batch} steps/epoch:{steps_per_epoch} total_steps:{total_steps}")
    if total_steps == 0:
        print('데이터/배치 설정을 확인하세요. 바로 학습을 시작합니다.')
        return True
    run_warm = min(warmup_steps, total_steps)
    print(f"벤치: {run_warm} step 실행하여 속도 측정합니다...")
    t0 = time.time()
    trainer.train(max_steps=run_warm)
    elapsed = time.time() - t0
    per_step = elapsed / run_warm
    est_total = per_step * total_steps
    est_h = est_total / 3600
    print(f"벤치 결과: {per_step:.2f}s/step → 예상 전체 소요: {est_h:.2f}시간")
    if est_h > max_hours:
        print(f"경고: 예상 소요시간이 {max_hours}시간({est_h:.2f}h) 초과합니다.")
        ans = input("계속 실행하시겠습니까? (y/N) ").strip().lower()
        return ans == 'y'
    return True

# ==========================================
# 1. 설정 (Configuration)
# ==========================================
MODEL_ID = "beomi/Llama-3-Open-Ko-8B-Instruct-preview"
DATA_PATH = "./news_data_for_train.csv"  # 전처리 완료된 데이터
OUTPUT_DIR = "./results/llama-3-news-lora"

BATCH_SIZE = 2
GRAD_ACCUMULATION = 8
LEARNING_RATE = 2e-4
NUM_EPOCHS = 1
MAX_SEQ_LENGTH = 1024 

# ==========================================
# 2. 모델 & 토크나이저 로드 (QLoRA)
# ==========================================
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    quantization_config=bnb_config,
    device_map="auto"
)

# ==========================================
# 3. 데이터셋 준비 (형식화 + 토큰화로 길이 제한)
def formatting_prompts_func(example):
    output_texts = []
    for text, summary in zip(example['cleaned_text'], example['ground_truth']):
        messages = [
            {"role": "system", "content": "뉴스 기사를 읽고 3문장 이내의 평어체(하라체)로 간결하게 요약하라."},
            {"role": "user", "content": f"기사:\n{text}"},
            {"role": "assistant", "content": summary}
        ]
        formatted = tokenizer.apply_chat_template(messages, tokenize=False)
        output_texts.append(formatted)
    return output_texts

def build_truncated_dataset(raw_ds, formatting_func, tokenizer, max_len):
    formatted_texts = []
    for ex in raw_ds:
        # formatting_func expects a dict-like example and returns list(s)
        # we pass single-example container
        formatted = formatting_func({"cleaned_text":[ex["cleaned_text"]], "ground_truth":[ex["ground_truth"]]})
        txt = formatted[0]
        tokenized = tokenizer(txt, truncation=True, max_length=max_len)
        truncated_txt = tokenizer.decode(tokenized["input_ids"], skip_special_tokens=False)
        formatted_texts.append(truncated_txt)
    return Dataset.from_dict({"text": formatted_texts})

# Replace original dataset variable with truncated dataset
dataset = load_dataset("csv", data_files=DATA_PATH, split="train")
dataset = build_truncated_dataset(dataset, formatting_prompts_func, tokenizer, MAX_SEQ_LENGTH)

# ==========================================
# 4. LoRA 및 Trainer 설정 (★ 수정됨)
# ==========================================
peft_config = LoraConfig(
    lora_alpha=16,
    lora_dropout=0.05,
    r=8,
    bias="none",
    task_type="CAUSAL_LM",
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"]
)

# ★ [수정 1] TrainingArguments 대신 SFTConfig 사용 & max_seq_length 포함
training_args = SFTConfig(
    output_dir=OUTPUT_DIR,
    per_device_train_batch_size=BATCH_SIZE,
    gradient_accumulation_steps=GRAD_ACCUMULATION,
    learning_rate=LEARNING_RATE,
    logging_steps=10,
    num_train_epochs=NUM_EPOCHS,
    max_grad_norm=0.3,
    warmup_ratio=0.03,
    lr_scheduler_type="cosine",
    fp16=True,
    optim="paged_adamw_32bit",
    save_strategy="epoch",
    report_to="none",
    # -- remove max_seq_length from here --
    dataset_text_field="text", # 필수값 (실제로는 formatting_func가 우선됨)
    packing=False
)

# 4. SFTTrainer 생성 — max_seq_length 제거
trainer = SFTTrainer(
    model=model,
    train_dataset=dataset,
    peft_config=peft_config,
    processing_class=tokenizer,
    args=training_args,
    formatting_func=None,   # 이미 포맷된 'text' 사용하므로 formatting_func는 None
)

# print("🚀 LLaMA-3 학습 시작...")
if bench_and_warn(trainer, dataset, BATCH_SIZE, GRAD_ACCUMULATION, NUM_EPOCHS):
    print("🚀 학습 진행합니다...")
    trainer.train()
else:
    print("학습을 취소했습니다. 설정(배치/에폭/데이터 수)을 조절하세요.")

trainer.model.save_pretrained(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)
print(f"✨ 학습 완료! 모델 저장 경로: {OUTPUT_DIR}")

In [ ]:
#테스트용 데이터 생성
import pandas as pd
import re
import os

# ==========================================
# 설정
# ==========================================
INPUT_FILE = "news_data_processed.csv"       # 아까 만든 1000개짜리 파일
OUTPUT_FILE = "news_data_for_train.csv"      # 학습에 들어갈 깨끗한 파일

# ==========================================
# 강력한 전처리 함수 (Deep Clean)
# ==========================================
def deep_clean_text(text):
    if not isinstance(text, str): return ""
    
    # 1. 숫자와 점 사이 줄바꿈/공백 복구 (1. \n 6 -> 1.6)
    text = re.sub(r'(\d+)\.\s+[\n\r]*(\d+)', r'\1.\2', text)
    
    # 2. 이메일 제거
    text = re.sub(r'[a-zA-Z0-9._%+-]+@[a-zA-Z0-9.-]+\.[a-zA-Z]{2,}', '', text)
    
    # 3. 기자 정보 및 바이라인 제거 (예: (서울=연합뉴스) OOO 기자 =)
    text = re.sub(r'\([=가-힣\s]*\)\s*[가-힣]{2,4}\s*기자\s*=', '', text)
    text = re.sub(r'\[.*?\]', '', text) # [사진=...] 제거
    
    # 4. 줄바꿈 및 다중 공백 제거
    text = re.sub(r'[\n\r]+', ' ', text)
    text = re.sub(r'\s+', ' ', text)
    
    return text.strip()

# ==========================================
# 메인 실행
# ==========================================
def main():
    # 1. 원본 파일 확인
    if not os.path.exists(INPUT_FILE):
        print(f"❌ '{INPUT_FILE}' 파일이 없습니다! 먼저 데이터 로드 코드를 실행해서 1000개 데이터를 만들어주세요.")
        return

    print("🧹 데이터 정제 시작...")
    df = pd.read_csv(INPUT_FILE)
    print(f"   - 원본 데이터 수: {len(df)}")
    
    # 2. 결측치 제거
    df = df.dropna(subset=['original_text', 'ground_truth'])
    
    # 3. 딥 클리닝 적용
    # 입력 데이터(기사)와 정답 데이터(요약문) 모두 깨끗하게 만듭니다.
    print("   - 텍스트 딥 클리닝 중...")
    df['cleaned_text'] = df['original_text'].apply(deep_clean_text)
    df['cleaned_truth'] = df['ground_truth'].apply(deep_clean_text)
    
    # 4. 너무 짧거나 비어있는 불량 데이터 삭제 (학습 방해 요소)
    df = df[df['cleaned_text'].str.len() > 50]  # 본문이 너무 짧은 것 제거
    df = df[df['cleaned_truth'].str.len() > 10] # 정답이 너무 짧은 것 제거
    
    print(f"   - 정제 후 최종 학습 데이터 수: {len(df)}")
    
    # 5. 저장 (컬럼명을 학습 코드에 맞게 정리)
    # 학습 코드에서는 'cleaned_text'와 'ground_truth'를 쓸 예정입니다.
    final_df = df[['cleaned_text', 'cleaned_truth']].rename(
        columns={'cleaned_truth': 'ground_truth'}
    )
    
    final_df.to_csv(OUTPUT_FILE, index=False, encoding='utf-8-sig')
    print(f"✨ 학습용 데이터 저장 완료: {OUTPUT_FILE}")
    print(f"👉 이제 'train_llama_fixed.py'를 실행하시면 됩니다!")

if __name__ == "__main__":
    main()

In [ ]:
#polyglot lora로 학습
import torch
import os
import gc
from datasets import load_dataset
from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    BitsAndBytesConfig,
    TrainingArguments  # ★ SFTConfig 대신 이거 씁니다
)
from peft import LoraConfig
from trl import SFTTrainer

# ==========================================
# 0. 메모리 초기화
# ==========================================
torch.cuda.empty_cache()
gc.collect()

# ==========================================
# 1. 설정
# ==========================================
MODEL_ID = "EleutherAI/polyglot-ko-5.8b"
DATA_PATH = "news_data_for_train.csv"
OUTPUT_DIR = "./results/polyglot-5.8b-news-lora"

BATCH_SIZE = 4
GRAD_ACCUMULATION = 4
LEARNING_RATE = 3e-4
NUM_EPOCHS = 1
MAX_SEQ_LENGTH = 1024

# ==========================================
# 2. 모델 & 토크나이저 로드
# ==========================================
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    quantization_config=bnb_config,
    device_map="auto"
)
model.config.use_cache = False

# ==========================================
# 3. 데이터셋 준비
# ==========================================
dataset = load_dataset("csv", data_files=DATA_PATH, split="train")

def formatting_prompts_func(example):
    output_texts = []
    for text, summary in zip(example['cleaned_text'], example['ground_truth']):
        prompt = f"기사: {text}\n\n전체 내용 요약: {summary}<|endoftext|>"
        output_texts.append(prompt)
    return output_texts

# ==========================================
# 4. LoRA 설정
# ==========================================
peft_config = LoraConfig(
    lora_alpha=32,
    lora_dropout=0.05,
    r=16,
    bias="none",
    task_type="CAUSAL_LM",
    target_modules=["query_key_value", "dense", "dense_h_to_4h", "dense_4h_to_h"]
)

# ==========================================
# 5. Trainer 설정 (★ 0.9.6 안정 버전용)
# ==========================================
training_args = TrainingArguments(  # ★ SFTConfig 대신 TrainingArguments 사용
    output_dir=OUTPUT_DIR,
    per_device_train_batch_size=BATCH_SIZE,
    gradient_accumulation_steps=GRAD_ACCUMULATION,
    learning_rate=LEARNING_RATE,
    logging_steps=10,
    num_train_epochs=NUM_EPOCHS,
    fp16=True,
    optim="paged_adamw_32bit",
    save_strategy="epoch",
    report_to="none",
    gradient_checkpointing=True,
    gradient_checkpointing_kwargs={"use_reentrant": False}
)

trainer = SFTTrainer(
    model=model,
    train_dataset=dataset,
    peft_config=peft_config,
    
    # ★ processing_class 대신 tokenizer 사용 (0.9.6 기준)
    tokenizer=tokenizer,
    
    args=training_args,
    formatting_func=formatting_prompts_func,
    
    # ★ max_seq_length를 여기에 넣으면 해결됨
    max_seq_length=MAX_SEQ_LENGTH,
)

print("🚀 Polyglot-5.8B 학습 시작 (Stable Version)...")
trainer.train()

model.config.use_cache = True
trainer.model.save_pretrained(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)
print(f"✨ 학습 완료! 모델 저장 경로: {OUTPUT_DIR}")

In [ ]:
#프롬프트 수정 없이 lora만 적용해 성능 비교
import pandas as pd
import torch
from transformers import (
    AutoTokenizer, 
    AutoModelForCausalLM, 
    BartForConditionalGeneration,
    PreTrainedTokenizerFast,
    BitsAndBytesConfig,
    AutoConfig
)
from peft import PeftModel
from rouge_score import rouge_scorer
import numpy as np
from tqdm import tqdm
import gc

# ==========================================
# 1. 설정
# ==========================================
TEST_DATA_PATH = "news_data_processed.csv" 
LLAMA_LORA_PATH = "./results/llama-3-news-lora"       
POLYGLOT_LORA_PATH = "./results/polyglot-5.8b-news-lora" 

LLAMA_BASE_ID = "beomi/Llama-3-Open-Ko-8B-Instruct-preview"
POLYGLOT_BASE_ID = "EleutherAI/polyglot-ko-5.8b"
BASELINE_MODEL_ID = "gogamza/kobart-summarization"

MAX_GEN_LEN = 150
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

# ==========================================
# 2. 유틸리티 함수
# ==========================================
def clear_gpu_memory():
    gc.collect()
    torch.cuda.empty_cache()
    print("🧹 GPU 메모리 청소 완료")

def calculate_rouge(preds, refs, model_name):
    scorer = rouge_scorer.RougeScorer(['rouge1', 'rouge2', 'rougeL'], use_stemmer=True)
    r1, r2, rl = [], [], []
    success_count = 0
    for pred, ref in zip(preds, refs):
        if not pred or len(pred.strip()) == 0: continue
        success_count += 1
        scores = scorer.score(ref, pred)
        r1.append(scores['rouge1'].fmeasure)
        r2.append(scores['rouge2'].fmeasure)
        rl.append(scores['rougeL'].fmeasure)
    return {
        "Model": model_name,
        "Success_Rate(%)": round((success_count / len(refs)) * 100, 2),
        "ROUGE-1": round(np.mean(r1) * 100, 2) if r1 else 0,
        "ROUGE-2": round(np.mean(r2) * 100, 2) if r2 else 0,
        "ROUGE-L": round(np.mean(rl) * 100, 2) if rl else 0
    }

# ==========================================
# 3. 모델별 평가 함수 (수정됨)
# ==========================================

def eval_llama(original_texts, ground_truths):
    print(f"\n🚀 [1/3] Llama-3 평가 시작...")
    clear_gpu_memory()
    try:
        bnb_config = BitsAndBytesConfig(
            load_in_4bit=True,
            bnb_4bit_use_double_quant=True,
            bnb_4bit_quant_type="nf4",
            bnb_4bit_compute_dtype=torch.float16
        )
        tokenizer = AutoTokenizer.from_pretrained(LLAMA_BASE_ID)
        tokenizer.pad_token = tokenizer.eos_token
        
        base_model = AutoModelForCausalLM.from_pretrained(
            LLAMA_BASE_ID,
            quantization_config=bnb_config,
            device_map="auto"
        )
        model = PeftModel.from_pretrained(base_model, LLAMA_LORA_PATH)
        model.eval()

        preds = []
        terminators = [tokenizer.eos_token_id, tokenizer.convert_tokens_to_ids("<|eot_id|>")]
        prompt_format = "다음 글을 요약해주세요:\n\n{}\n\n요약:"

        for text in tqdm(original_texts, desc="Generating Llama"):
            if len(text) > 1500: text = text[:1500]
            input_text = prompt_format.format(text)
            inputs = tokenizer(input_text, return_tensors="pt", max_length=1024, truncation=True).to(DEVICE)
            
            with torch.no_grad():
                outputs = model.generate(
                    **inputs, 
                    max_new_tokens=MAX_GEN_LEN,
                    num_beams=2,
                    do_sample=False,
                    repetition_penalty=1.2,
                    eos_token_id=terminators,
                    pad_token_id=tokenizer.eos_token_id
                )
                gen_text = tokenizer.decode(outputs[0][inputs['input_ids'].shape[1]:], skip_special_tokens=True)
                preds.append(gen_text.strip())

        del model, base_model, tokenizer
        return calculate_rouge(preds, ground_truths, 'Llama-3 (LoRA)')
    except Exception as e:
        print(f"❌ Llama-3 에러 발생: {e}")
        return None

def eval_polyglot(original_texts, ground_truths):
    print(f"\n🚀 [2/3] Polyglot 평가 시작...")
    clear_gpu_memory()
    try:
        bnb_config = BitsAndBytesConfig(
            load_in_4bit=True,
            bnb_4bit_use_double_quant=True,
            bnb_4bit_quant_type="nf4",
            bnb_4bit_compute_dtype=torch.float16
        )
        tokenizer = AutoTokenizer.from_pretrained(POLYGLOT_BASE_ID)
        base_model = AutoModelForCausalLM.from_pretrained(
            POLYGLOT_BASE_ID,
            quantization_config=bnb_config,
            device_map="auto"
        )
        model = PeftModel.from_pretrained(base_model, POLYGLOT_LORA_PATH)
        model.eval()

        preds = []
        prompt_format = "전문: {}\n\n요약:"

        for text in tqdm(original_texts, desc="Generating Polyglot"):
            if len(text) > 1500: text = text[:1500]
            input_text = prompt_format.format(text)
            inputs = tokenizer(input_text, return_tensors="pt", max_length=1024, truncation=True).to(DEVICE)
            
            # [수정 1] token_type_ids 제거 (Polyglot 에러 해결)
            if 'token_type_ids' in inputs:
                del inputs['token_type_ids']
            
            with torch.no_grad():
                outputs = model.generate(
                    **inputs, 
                    max_new_tokens=MAX_GEN_LEN,
                    num_beams=2,
                    do_sample=False,
                    repetition_penalty=1.2,
                    pad_token_id=tokenizer.eos_token_id
                )
                gen_text = tokenizer.decode(outputs[0][inputs['input_ids'].shape[1]:], skip_special_tokens=True)
                preds.append(gen_text.strip())
        
        del model, base_model, tokenizer
        return calculate_rouge(preds, ground_truths, 'Polyglot (LoRA)')
    except Exception as e:
        print(f"❌ Polyglot 에러 발생: {e}")
        return None

def eval_bart(original_texts, ground_truths):
    print(f"\n🚀 [3/3] Baseline (BART) 평가 시작...")
    clear_gpu_memory()
    try:
        # [수정 2] delattr 대신 속성 덮어쓰기 (BART 에러 해결)
        config = AutoConfig.from_pretrained(BASELINE_MODEL_ID)
        # 삭제(del)가 안되므로, 기본값으로 덮어씁니다.
        config.num_labels = 0 
        config.id2label = {0: "LABEL_0"}
        config.label2id = {"LABEL_0": 0}

        tokenizer = PreTrainedTokenizerFast.from_pretrained(BASELINE_MODEL_ID)
        # config를 명시적으로 전달
        model = BartForConditionalGeneration.from_pretrained(BASELINE_MODEL_ID, config=config).to(DEVICE)
        model.eval()

        preds = []
        for text in tqdm(original_texts, desc="Generating BART"):
            inputs = tokenizer(text, return_tensors="pt", max_length=1024, truncation=True).to(DEVICE)
            with torch.no_grad():
                outputs = model.generate(
                    **inputs, 
                    max_length=MAX_GEN_LEN, 
                    num_beams=4, 
                    length_penalty=2.0, 
                    early_stopping=True
                )
                gen_text = tokenizer.decode(outputs[0], skip_special_tokens=True)
                preds.append(gen_text.strip())
        
        del model, tokenizer
        return calculate_rouge(preds, ground_truths, 'Baseline (BART)')
    except Exception as e:
        print(f"❌ BART 에러 발생: {e}")
        return None

# ==========================================
# 4. 메인 실행
# ==========================================
def main():
    print(f"📂 데이터 로드: {TEST_DATA_PATH}")
    df = pd.read_csv(TEST_DATA_PATH)
    if len(df) > 100: df = df.head(100)
    
    original_texts = df['original_text'].astype(str).tolist()
    ground_truths = df['ground_truth'].astype(str).tolist()
    results = []

    res_llama = eval_llama(original_texts, ground_truths)
    if res_llama: results.append(res_llama)
    
    res_polyglot = eval_polyglot(original_texts, ground_truths)
    if res_polyglot: results.append(res_polyglot)
    
    res_bart = eval_bart(original_texts, ground_truths)
    if res_bart: results.append(res_bart)

    if results:
        final_df = pd.DataFrame(results)
        final_df = final_df[['Model', 'Success_Rate(%)', 'ROUGE-1', 'ROUGE-2', 'ROUGE-L']]
        print("\n" + "="*50)
        print("🏆 최종 성능 평가 결과")
        print("="*50)
        print(final_df)
        final_df.to_csv("final_lora_evaluation_result.csv", index=False)
    else:
        print("❌ 생성된 결과가 없습니다.")

if __name__ == "__main__":
    main()

In [ ]:
#llama 맞춤 프롬프트 수정 후 성능 평가 진행
import pandas as pd
import torch
from transformers import (
    AutoTokenizer, 
    AutoModelForCausalLM, 
    BartForConditionalGeneration,
    PreTrainedTokenizerFast,
    BitsAndBytesConfig,
    AutoConfig
)
from peft import PeftModel
from rouge_score import rouge_scorer
import numpy as np
from tqdm import tqdm
import gc

# ==========================================
# 1. 설정
# ==========================================
TEST_DATA_PATH = "news_data_processed.csv" 
LLAMA_LORA_PATH = "./results/llama-3-news-lora"       
POLYGLOT_LORA_PATH = "./results/polyglot-5.8b-news-lora" 

LLAMA_BASE_ID = "beomi/Llama-3-Open-Ko-8B-Instruct-preview"
POLYGLOT_BASE_ID = "EleutherAI/polyglot-ko-5.8b"
BASELINE_MODEL_ID = "gogamza/kobart-summarization"

MAX_GEN_LEN = 150
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

# ==========================================
# 2. 유틸리티 함수
# ==========================================
def clear_gpu_memory():
    gc.collect()
    torch.cuda.empty_cache()
    print("🧹 GPU 메모리 청소 완료")

def calculate_rouge(preds, refs, model_name):
    scorer = rouge_scorer.RougeScorer(['rouge1', 'rouge2', 'rougeL'], use_stemmer=True)
    r1, r2, rl = [], [], []
    success_count = 0
    for pred, ref in zip(preds, refs):
        if not pred or len(pred.strip()) == 0: continue
        success_count += 1
        scores = scorer.score(ref, pred)
        r1.append(scores['rouge1'].fmeasure)
        r2.append(scores['rouge2'].fmeasure)
        rl.append(scores['rougeL'].fmeasure)
    return {
        "Model": model_name,
        "Success_Rate(%)": round((success_count / len(refs)) * 100, 2),
        "ROUGE-1": round(np.mean(r1) * 100, 2) if r1 else 0,
        "ROUGE-2": round(np.mean(r2) * 100, 2) if r2 else 0,
        "ROUGE-L": round(np.mean(rl) * 100, 2) if rl else 0
    }

# ==========================================
# 3. 모델별 평가 함수 (수정됨)
# ==========================================
def apply_llama3_template(text):
    return (
        "<|begin_of_text|>"
        "<|start_header_id|>system<|end_header_id|>\n\n"
        "You are a helpful AI assistant. Please summarize the following Korean text."
        "<|eot_id|>"
        "<|start_header_id|>user<|end_header_id|>\n\n"
        f"다음 뉴스 기사를 핵심만 요약해주세요:\n\n{text}"
        "<|eot_id|>"
        "<|start_header_id|>assistant<|end_header_id|>\n\n"
    )
def eval_llama(original_texts, ground_truths):
    print(f"\n🚀 [1/3] Llama-3 (Chat Template 적용) 평가 시작...")
    clear_gpu_memory()
    
    try:
        bnb_config = BitsAndBytesConfig(
            load_in_4bit=True,
            bnb_4bit_use_double_quant=True,
            bnb_4bit_quant_type="nf4",
            bnb_4bit_compute_dtype=torch.float16
        )

        tokenizer = AutoTokenizer.from_pretrained(LLAMA_BASE_ID)
        tokenizer.pad_token = tokenizer.eos_token
        
        base_model = AutoModelForCausalLM.from_pretrained(
            LLAMA_BASE_ID,
            quantization_config=bnb_config,
            device_map="auto"
        )
        model = PeftModel.from_pretrained(base_model, LLAMA_LORA_PATH)
        model.eval()

        preds = []
        # LLaMA-3 전용 종료 토큰들
        terminators = [
            tokenizer.eos_token_id,
            tokenizer.convert_tokens_to_ids("<|eot_id|>")
        ]

        for text in tqdm(original_texts, desc="Generating Llama"):
            if len(text) > 1500: text = text[:1500]
            
            # [핵심 변경] 단순 format 대신 LLaMA-3 전용 템플릿 적용
            input_text = apply_llama3_template(text)
            
            inputs = tokenizer(input_text, return_tensors="pt", max_length=2048, truncation=True).to(DEVICE)
            
            with torch.no_grad():
                outputs = model.generate(
                    **inputs, 
                    max_new_tokens=150,
                    num_beams=2,
                    do_sample=False,
                    repetition_penalty=1.2, # 반복 방지 페널티
                    eos_token_id=terminators, # 종료 토큰 확실히 지정
                    pad_token_id=tokenizer.eos_token_id
                )
                # 프롬프트 길이만큼 자르고 생성된 부분만 디코딩
                gen_text = tokenizer.decode(outputs[0][inputs['input_ids'].shape[1]:], skip_special_tokens=True)
                preds.append(gen_text.strip())

        del model, base_model, tokenizer
        return calculate_rouge(preds, ground_truths, 'Llama-3 (LoRA/Chat)')

    except Exception as e:
        print(f"❌ Llama-3 에러 발생: {e}")
        return None

def eval_polyglot(original_texts, ground_truths):
    print(f"\n🚀 [2/3] Polyglot 평가 시작...")
    clear_gpu_memory()
    try:
        bnb_config = BitsAndBytesConfig(
            load_in_4bit=True,
            bnb_4bit_use_double_quant=True,
            bnb_4bit_quant_type="nf4",
            bnb_4bit_compute_dtype=torch.float16
        )
        tokenizer = AutoTokenizer.from_pretrained(POLYGLOT_BASE_ID)
        base_model = AutoModelForCausalLM.from_pretrained(
            POLYGLOT_BASE_ID,
            quantization_config=bnb_config,
            device_map="auto"
        )
        model = PeftModel.from_pretrained(base_model, POLYGLOT_LORA_PATH)
        model.eval()

        preds = []
        prompt_format = "전문: {}\n\n요약:"

        for text in tqdm(original_texts, desc="Generating Polyglot"):
            if len(text) > 1500: text = text[:1500]
            input_text = prompt_format.format(text)
            inputs = tokenizer(input_text, return_tensors="pt", max_length=1024, truncation=True).to(DEVICE)
            
            # [수정 1] token_type_ids 제거 (Polyglot 에러 해결)
            if 'token_type_ids' in inputs:
                del inputs['token_type_ids']
            
            with torch.no_grad():
                outputs = model.generate(
                    **inputs, 
                    max_new_tokens=MAX_GEN_LEN,
                    num_beams=2,
                    do_sample=False,
                    repetition_penalty=1.2,
                    pad_token_id=tokenizer.eos_token_id
                )
                gen_text = tokenizer.decode(outputs[0][inputs['input_ids'].shape[1]:], skip_special_tokens=True)
                preds.append(gen_text.strip())
        
        del model, base_model, tokenizer
        return calculate_rouge(preds, ground_truths, 'Polyglot (LoRA)')
    except Exception as e:
        print(f"❌ Polyglot 에러 발생: {e}")
        return None

def eval_bart(original_texts, ground_truths):
    print(f"\n🚀 [3/3] Baseline (BART) 평가 시작...")
    clear_gpu_memory()
    try:
        # [수정 2] delattr 대신 속성 덮어쓰기 (BART 에러 해결)
        config = AutoConfig.from_pretrained(BASELINE_MODEL_ID)
        # 삭제(del)가 안되므로, 기본값으로 덮어씁니다.
        config.num_labels = 0 
        config.id2label = {0: "LABEL_0"}
        config.label2id = {"LABEL_0": 0}

        tokenizer = PreTrainedTokenizerFast.from_pretrained(BASELINE_MODEL_ID)
        # config를 명시적으로 전달
        model = BartForConditionalGeneration.from_pretrained(BASELINE_MODEL_ID, config=config).to(DEVICE)
        model.eval()

        preds = []
        for text in tqdm(original_texts, desc="Generating BART"):
            inputs = tokenizer(text, return_tensors="pt", max_length=1024, truncation=True).to(DEVICE)
            with torch.no_grad():
                outputs = model.generate(
                    **inputs, 
                    max_length=MAX_GEN_LEN, 
                    num_beams=4, 
                    length_penalty=2.0, 
                    early_stopping=True
                )
                gen_text = tokenizer.decode(outputs[0], skip_special_tokens=True)
                preds.append(gen_text.strip())
        
        del model, tokenizer
        return calculate_rouge(preds, ground_truths, 'Baseline (BART)')
    except Exception as e:
        print(f"❌ BART 에러 발생: {e}")
        return None

# ==========================================
# 4. 메인 실행
# ==========================================
def main():
    print(f"📂 데이터 로드: {TEST_DATA_PATH}")
    df = pd.read_csv(TEST_DATA_PATH)
    if len(df) > 100: df = df.head(100)
    
    original_texts = df['original_text'].astype(str).tolist()
    ground_truths = df['ground_truth'].astype(str).tolist()
    results = []

    res_llama = eval_llama(original_texts, ground_truths)
    if res_llama: results.append(res_llama)
    
    res_polyglot = eval_polyglot(original_texts, ground_truths)
    if res_polyglot: results.append(res_polyglot)
    
    res_bart = eval_bart(original_texts, ground_truths)
    if res_bart: results.append(res_bart)

    if results:
        final_df = pd.DataFrame(results)
        final_df = final_df[['Model', 'Success_Rate(%)', 'ROUGE-1', 'ROUGE-2', 'ROUGE-L']]
        print("\n" + "="*50)
        print("🏆 최종 성능 평가 결과")
        print("="*50)
        print(final_df)
        final_df.to_csv("final_lora_evaluation_result.csv", index=False)
    else:
        print("❌ 생성된 결과가 없습니다.")

if __name__ == "__main__":
    main()